In [1]:
## Quantum Computing imports
import pennylane as qml
import numpy as np
from math import pi

## GNN imports
import torch
from torch_geometric.data import Data, DataLoader
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, global_mean_pool, EdgeConv
from sklearn.metrics import accuracy_score, roc_auc_score

## Plotting imports
import matplotlib.pyplot as plt

import os
import random
import urllib.request

#import tools
from qgnn_hybrid.data import get_dataloaders
import qgnn_hybrid.utils as nbtools
from qgnn_hybrid.models import *



/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

train_loader, val_loader, test_loader = get_dataloaders(batch_size=32)
in_channels = train_loader.dataset[0].x.shape[1]

Dataset already exists.


/Users/alexcampbell/Documents/QGNN-HybridGNN-QNN/notebook_tools.py:50: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  pt  = (pt - pt.mean()) / (pt.std() + 1e-6)
/Users/alexcampbell/Documents/QGNN-HybridGNN-QNN/notebook_tools.py:51: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  eta = (eta - eta.mean()) / (eta.std() + 1e-6)


In [3]:
AUCs = []
AUC_qnn_basic_42 = 0.8604
for seed in [42, 43, 44, 45, 46]:
    print(f"Starting Seed: {seed}")
    set_seed(seed)
    in_channels = train_loader.dataset[0].x.shape[1]
    model = HybridGNN_MLP(in_channels=in_channels)
    model, history = nbtools.train_model(
    model,
    train_loader,
    val_loader=val_loader,   # or split a validation set
    epochs=100,
    patience=5,
    save_every=1,
    model_name=f"GNN_MLP_Seed_{seed}_best_model",
    training_history=f"GNN_MLP_Seed_{seed}_training_history",
    checkpoint_path=f"GNN_MLP_Seed_{seed}_checkpoint",
    )
    acc, auc = nbtools.evaluate_model(model, test_loader)
    AUCs.append(auc)
mean_auc = np.mean(AUCs)
std_auc = np.std(AUCs)
print(f"Mean AUC: {mean_auc} Std: {std_auc}")

Starting Seed: 42
[10:15:55] Epoch 1 | Train Loss: 0.5288 | Val Loss: 0.4911 | Val Acc: 0.7697 | Val AUC: 0.8429 | Time: 54.41s
✅ Saved best model
[10:16:30] Epoch 2 | Train Loss: 0.4969 | Val Loss: 0.4854 | Val Acc: 0.7751 | Val AUC: 0.8486 | Time: 34.65s
✅ Saved best model
[10:17:03] Epoch 3 | Train Loss: 0.4899 | Val Loss: 0.4837 | Val Acc: 0.7778 | Val AUC: 0.8504 | Time: 32.55s
✅ Saved best model
[10:17:32] Epoch 4 | Train Loss: 0.4869 | Val Loss: 0.4881 | Val Acc: 0.7698 | Val AUC: 0.8512 | Time: 28.97s
[10:18:01] Epoch 5 | Train Loss: 0.4845 | Val Loss: 0.4798 | Val Acc: 0.7778 | Val AUC: 0.8534 | Time: 29.09s
✅ Saved best model
[10:18:31] Epoch 6 | Train Loss: 0.4834 | Val Loss: 0.4864 | Val Acc: 0.7737 | Val AUC: 0.8522 | Time: 29.90s
[10:19:00] Epoch 7 | Train Loss: 0.4817 | Val Loss: 0.4755 | Val Acc: 0.7789 | Val AUC: 0.8551 | Time: 29.36s
✅ Saved best model
[10:19:29] Epoch 8 | Train Loss: 0.4800 | Val Loss: 0.4734 | Val Acc: 0.7795 | Val AUC: 0.8561 | Time: 29.22s
✅ Saved

In [3]:
AUCs = []
AUC_qnn_basic_42 = 0.8604
for seed in [42, 43, 44, 45, 46]:
    print(f"Starting Seed: {seed}")
    set_seed(seed)
    #train_loader, val_loader, test_loader = get_dataloaders(batch_size=32, seed=seed)
    in_channels = train_loader.dataset[0].x.shape[1]
    model = HybridGNN_QNN_basic_torch(in_channels=in_channels)
    model, history = nbtools.train_model(
    model,
    train_loader,
    val_loader=val_loader,   # or split a validation set
    epochs=100,
    patience=5,
    save_every=1,
    model_name=f"GNN_QNN_Basic_Torch_Seed_{seed}_best_model",
    training_history=f"GNN_QNN_Basic_Torch_Seed_{seed}_training_history",
    checkpoint_path=f"GNN_QNN_Basic_Torch_Seed_{seed}_checkpoint",
    )
    acc, auc = nbtools.evaluate_model(model, test_loader)
    AUCs.append(auc)
mean_auc = np.mean(AUCs)
std_auc = np.std(AUCs)
print(f"Mean AUC: {mean_auc} Std: {std_auc}")

Starting Seed: 42
[15:23:45] Epoch 1 | Train Loss: 0.5470 | Val Loss: 0.6391 | Val Acc: 0.7048 | Val AUC: 0.8438 | Time: 184.00s
✅ Saved best model
[15:26:28] Epoch 2 | Train Loss: 0.5037 | Val Loss: 0.4823 | Val Acc: 0.7759 | Val AUC: 0.8495 | Time: 162.85s
✅ Saved best model
[15:29:08] Epoch 3 | Train Loss: 0.4973 | Val Loss: 0.4829 | Val Acc: 0.7721 | Val AUC: 0.8492 | Time: 160.29s
[15:31:44] Epoch 4 | Train Loss: 0.4920 | Val Loss: 0.4893 | Val Acc: 0.7725 | Val AUC: 0.8538 | Time: 155.91s
[15:34:21] Epoch 5 | Train Loss: 0.4898 | Val Loss: 0.5274 | Val Acc: 0.7436 | Val AUC: 0.8517 | Time: 156.77s
[15:36:55] Epoch 6 | Train Loss: 0.4869 | Val Loss: 0.4858 | Val Acc: 0.7783 | Val AUC: 0.8546 | Time: 154.19s
[15:39:32] Epoch 7 | Train Loss: 0.4840 | Val Loss: 0.4745 | Val Acc: 0.7806 | Val AUC: 0.8552 | Time: 157.37s
✅ Saved best model
[15:42:07] Epoch 8 | Train Loss: 0.4830 | Val Loss: 0.4803 | Val Acc: 0.7755 | Val AUC: 0.8547 | Time: 154.47s
[15:44:42] Epoch 9 | Train Loss: 0.48

In [ ]:
QNN_AUCs = AUCs
MLP_AUCs = [0.8561, 0.8601, 0.8629, 0.8632, 0.8624]

from scipy.stats import ttest_rel

t_stat, p_val = ttest_rel(MLP_AUCs, QNN_AUCs)
print(p_val)


[0.8563336737191745,
 0.8625470276235234,
 0.8636517844712934,
 0.8612881397196637,
 0.8600059445533349]

In [18]:


# ---- your functions ----
def _bc(val, ndim):
    if val.dim() == 0:
        return val
    return val.reshape(-1, *([1] * ndim))


def apply_ry(re, im, theta, qubit, nq):
    print("re before first transformation", re)
    re = re.movedim(qubit + 1, -1)
    print("re after first transformation", re)
    im = im.movedim(qubit + 1, -1)
    c = _bc(torch.cos(theta / 2), nq - 1)
    s = _bc(torch.sin(theta / 2), nq - 1)
    print("angles", c, s)
    re0, re1 = re[..., 0], re[..., 1]
    im0, im1 = im[..., 0], im[..., 1]
    new_re = torch.stack([c * re0 - s * re1, s * re0 + c * re1], dim=-1)
    new_im = torch.stack([c * im0 - s * im1, s * im0 + c * im1], dim=-1)
    return new_re.movedim(-1, qubit + 1), new_im.movedim(-1, qubit + 1)


# ---- helper to print state ----
def print_state(re, im, title=""):
    probs = (re**2 + im**2).reshape(re.shape[0], -1)
    print(f"\n{title}")
    print("State probabilities:")
    print(probs)

def apply_cnot(re, im, control, target, nq):
    ## Move Control qubit to second to last axis
    re = re.movedim(control + 1, -2)
    im = im.movedim(control + 1, -2)

    ## Adjust target index after move
    t_ax = (target + 1) if target < control else target

    ## Move Target to last axis
    re = re.movedim(t_ax, -1)
    im = im.movedim(t_ax, -1)

    ## Split control states
    re_c0, re_c1 = re[..., 0, :], re[..., 1, :]
    im_c0, im_c1 = im[..., 0, :], im[..., 1, :]

    re_t0, re_t1 = re_c1[..., 0], re_c1[..., 1]
    im_t0, im_t1 = im_c1[..., 0], im_c1[..., 1]

    ## Flip target when control = 1
    new_re_c1 = torch.stack([re_t1, re_t0], dim=-1)
    new_im_c1 = torch.stack([im_t1, im_t0], dim=-1)

    ## Recombine
    out_re = torch.stack([re_c0, new_re_c1], dim=-2)
    out_im = torch.stack([im_c0, new_im_c1], dim=-2)

    ## Move axes back
    out_re = out_re.movedim(-1, t_ax).movedim(-1, control + 1)
    out_im = out_im.movedim(-1, t_ax).movedim(-1, control + 1)
    return out_re, out_im


# # ---- setup a simple 2-qubit system ----
# B = 1        # batch size
# nq = 2       # 2 qubits

# # Start in |00>
# re = torch.zeros(B, 2, 2)
# im = torch.zeros_like(re)

# re[:, 1, 0] = 1.0  # amplitude of |00>
# print_state(re, im, "Initial state |00>")

# # ---- apply RY to qubit 0 ----
# theta = torch.tensor([torch.pi / 2])  # 90 degrees

# re, im = apply_cnot(re, im, 0,1,2)

# print_state(re, im, "After CNOT on qubit 0")

# |10> state
re = torch.zeros(1, 2, 2)
im = torch.zeros_like(re)
re[0, 1, 0] = 1.0
print(re)
# Apply CNOT (control=0, target=1)
re, im = apply_cnot(re, im, control=0, target=1, nq=2)

print((re**2).reshape(1, -1))

tensor([[[0., 0.],
         [1., 0.]]])
tensor([[0., 1., 0., 0.]])
